In [1]:
import os
from pathlib import Path
from openai import OpenAI
import sys
from dotenv import load_dotenv
import json

In [2]:
openai = OpenAI()

In [3]:
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")
if not api_key:
    raise ValueError("OPENROUTER_API_KEY is missing from your .env file.")

In [4]:
model = "openai/gpt-oss-120b"

In [5]:
# add the parent directory into path to import packages
sys.path.append(str(Path.cwd().parent.parent))

from Problems.Text. text_prompts import prompts 

print(prompts)

{'regular_system_prompt': 'You are a physics expert. Based on the question that is given solve the problem. The question has equations and symbols in latex, but your answer should not be in latex.', 'detail_system_prompt': 'You are a physics expert. Based on the question that is given solve the problem, but if there are any physical laws broken then mention it and also say if the problem can be solved or not. The question has equations and symbols in latex, but your answer should not be in latex.'}


In [6]:
print(prompts.keys())

dict_keys(['regular_system_prompt', 'detail_system_prompt'])


In [7]:
# Path to markdown files
text_folder = Path("../../Problems/Text")

# List Markdown filenames, including the .md extension
file_names = sorted(file.name for file in text_folder.glob("*.md"))

print("Total number of files:::",len(file_names))
print(file_names)

Total number of files::: 9
['Problem_1_Obvious_Q.md', 'Problem_1_Regular_Q.md', 'Problem_1_non_obvious_Q.md', 'Problem_2_Obvious_Q.md', 'Problem_2_Regular_Q.md', 'Problem_2_non_obvious_Q.md', 'Problem_3_Obvious_Q.md', 'Problem_3_Regular_Q.md', 'Problem_3_non_obvious_Q.md']


In [8]:
## connect to OpenRouter

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
)

In [9]:
folder = Path.cwd().parent.parent
print(folder)

c:\Users\Shisir\Desktop\Physics_LLM_Project


In [10]:
%%time

meta_data= {} ## useful for creating metadata stored as a json file. Currently an empty dictionary

for keys in prompts:
    for file in file_names:
        text_folder = Path("../../Problems/Text")   ## first read the files
        file_path = text_folder/ file
        
        if keys == "detail_system_prompt" and "regular" in file_path.stem.lower():  ## skip if keys have detail_system_prompt and 
            continue     ## problems that have regular in their name. We dont need Regular problems to use detail prompts but run the rest
                                                                    
        problem_text = file_path.read_text(encoding="utf-8")    

        response = client.chat.completions.create(
                    model="openai/gpt-oss-120b",
                    messages=[
                        {"role": "system", "content": prompts[keys]},
                        {"role": "user", "content": problem_text},
                            ],
                        )
        model=model
        model_name=model.split("/")[-1]
        output_text = response.choices[0].message.content

        output_filename= f"{model_name}_{file_path.stem}_{keys}.md"
        folder = Path.cwd().parent.parent / "Result" / "open_weight_models" / "Text"
        output_file_path = folder / output_filename
        output_file_path.write_text(output_text, encoding="utf-8")
           
        ## Create metadata
        
        output_name = f"{model_name}_{file_path.stem}_{keys}.json" ## create filename
        output_folder = Path.cwd().parent.parent / "Result" / "open_weight_models" / "Text" / "metadata"
        
        # get token usage (may not be provided by every API)
        usage = getattr(response, "usage", None)
        metadata = {                         
                        "requested_model": model,
                        "returned_model": getattr(response, "model", None),
                        "response_id": getattr(response, "id", None),
        
                        "problem_text": problem_text,
                        "prompt_name": keys,
                        "system_prompt": prompts[keys],
                        "generation_settings": meta_data,
        
        
                        "token_usage": {
                            "input_tokens": getattr(usage, "prompt_tokens", None),
                            "output_tokens": getattr(usage, "completion_tokens", None),
                            "total_tokens": getattr(usage, "total_tokens", None),
                        },
                    }
        
        ## convert dictionary to json
        
        meta_data_file_path = output_folder / output_name
        meta_data_file_path.write_text(
                        json.dumps(metadata, indent=4, ensure_ascii=False),
                        encoding="utf-8",
                    )
        print(f"Completed:::{model_name}_{file_path.stem}_{keys}")    


Completed:::gpt-oss-120b_Problem_1_Obvious_Q_regular_system_prompt
Completed:::gpt-oss-120b_Problem_1_Regular_Q_regular_system_prompt
Completed:::gpt-oss-120b_Problem_1_non_obvious_Q_regular_system_prompt
Completed:::gpt-oss-120b_Problem_2_Obvious_Q_regular_system_prompt
Completed:::gpt-oss-120b_Problem_2_Regular_Q_regular_system_prompt
Completed:::gpt-oss-120b_Problem_2_non_obvious_Q_regular_system_prompt
Completed:::gpt-oss-120b_Problem_3_Obvious_Q_regular_system_prompt
Completed:::gpt-oss-120b_Problem_3_Regular_Q_regular_system_prompt
Completed:::gpt-oss-120b_Problem_3_non_obvious_Q_regular_system_prompt
Completed:::gpt-oss-120b_Problem_1_Obvious_Q_detail_system_prompt
Completed:::gpt-oss-120b_Problem_1_non_obvious_Q_detail_system_prompt
Completed:::gpt-oss-120b_Problem_2_Obvious_Q_detail_system_prompt
Completed:::gpt-oss-120b_Problem_2_non_obvious_Q_detail_system_prompt
Completed:::gpt-oss-120b_Problem_3_Obvious_Q_detail_system_prompt
Completed:::gpt-oss-120b_Problem_3_non_obvious_